# 🔍 LogSeer Token Importance

Shows which tokens most influenced a tree-based model's predictions, ranked by feature importance. Works with any sklearn model that exposes `feature_importances_` (XGBoost, Random Forest, Gradient Boosting).

If you already trained one of those models on this machine, `TOKENIZER_FILE` and `MODEL_FILE` are already generated. Otherwise use the Colab cell to load them from Google Drive, or manually copying them to `MODEL_BASE`.

 ---

> **▶️ Start here** — run all cells from top to bottom. Skip any marked **COLAB ONLY** if running on a self-hosted environment.

In [ ]:
# Setup
import os
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

In [ ]:
# Configuration
MODEL_BASE     = './'                       # directory where trained models are stored
TOKENIZER_FILE = 'tokenizer.pickle'
MODEL_FILE       = 'xgb.pkl'
TOKENIZER_PATH = MODEL_BASE + TOKENIZER_FILE
XGB_FILE_PATH  = MODEL_BASE + MODEL_FILE
TOP_N          = 50

---
### ☁️ COLAB ONLY — run the cell below

> Copy trained models from Google Drive. **Skip if running on a self-hosted environment.**

---

In [ ]:
# Copy trained models from Google Drive.
from google.colab import drive
import shutil
drive.mount('/content/drive')

DRIVE_BASE      = '/content/drive/MyDrive/Colab Notebooks/logseer/'
DRIVE_MODEL_DIR = DRIVE_BASE + '/models/'

shutil.copy(DRIVE_MODEL_DIR + TOKENIZER_FILE, MODEL_BASE + TOKENIZER_FILE)
shutil.copy(DRIVE_MODEL_DIR + MODEL_FILE,       MODEL_BASE + MODEL_FILE)

> Model files are now ready. The cells below compute token importance scores from the model, print the top `TOP_N` tokens as a table, and plot them as a bar chart.

In [ ]:
# Load tokenizer and model; build a reverse mapping from token ID to word.
with open(MODEL_BASE + TOKENIZER_FILE, 'rb') as f:
    tokenizer = pickle.load(f)

with open(MODEL_BASE + MODEL_FILE, 'rb') as f:
    model = pickle.load(f)

# word_index is token -> int, invert to int -> token
index_to_word = {v: k for k, v in tokenizer.word_index.items()}
print(f'Vocabulary size: {len(index_to_word)}')

In [ ]:
# Extract non-zero feature importance scores and map each column index back to its token name.
scores = model.feature_importances_

# texts_to_matrix columns correspond to token indices 0..MAX_NB_WORDS
# index 0 is unused padding, so token i maps to column i
rows = []
for col_idx, score in enumerate(scores):
    if score > 0 and col_idx in index_to_word:
        rows.append({'token': index_to_word[col_idx], 'importance': score})

df = pd.DataFrame(rows).sort_values('importance', ascending=False).reset_index(drop=True)
print(f'Non-zero features: {len(df)}')
df.head(TOP_N)

In [ ]:
# Plot the top N tokens as a horizontal bar chart ranked by importance.
top = df.head(TOP_N)
plt.figure(figsize=(10, TOP_N * 0.3))
plt.barh(top['token'][::-1], top['importance'][::-1])
plt.xlabel('Importance')
plt.title(f'Top {TOP_N} tokens by feature importance')
plt.tight_layout()
plt.show()